# Silver: Produtos

**Objetivo:** transformar a tabela Bronze de Produtos (cópia fiel, sem tratamento) em uma tabela Silver confiável e versionada, aplicando:
1. Casting de preços (texto → decimal)
2. Controle SCD Tipo 2 real, via MERGE (compara com a versão vigente; se houver mudança, fecha a versão antiga e abre uma nova)

**Fonte:** tabela Delta `bronze.produtos`.

**Destino:** tabela Delta `silver.produtos`.

**Dependências:** nenhuma classe de `utils` reutilizada nesta primeira versão — a lógica de MERGE será escrita diretamente aqui, e poderá ser extraída para uma nova versão do `SCD2Handler` caso o padrão se repita de forma idêntica nos próximos notebooks (Lojas, Representantes).

In [0]:
# Imports e configuração do schema

from pyspark.sql.functions import col, regexp_replace

spark.sql("CREATE SCHEMA IF NOT EXISTS poc_latam_food.silver")

print("Schema 'silver' verificado/criado com sucesso.")

In [0]:
# Casting dos preços, derivação do SKU, e padronização de texto

from pyspark.sql.functions import concat_ws, upper, translate

caracteres_com_acento = "áàãâäéèêëíìîïóòõôöúùûüçñÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇÑ"
caracteres_sem_acento = "aaaaaeeeeiiiiooooouuuucnAAAAAEEEEIIIIOOOOOUUUUCN"

df_bronze_produtos = spark.table("poc_latam_food.bronze.produtos")

df_produtos_casted = (
    df_bronze_produtos
    .withColumn("preco_brasil_brl", regexp_replace(col("preco_brasil_brl"), ",", ".").cast("decimal(10,2)"))
    .withColumn("preco_argentina_ars", regexp_replace(col("preco_argentina_ars"), ",", ".").cast("decimal(10,2)"))
    .withColumn("preco_mexico_mxn", regexp_replace(col("preco_mexico_mxn"), ",", ".").cast("decimal(10,2)"))
    .withColumn("sku", upper(concat_ws("_", col("nome_interno"), col("tamanho"))))
    .withColumn("nome_brasil", upper(translate(col("nome_brasil"), caracteres_com_acento, caracteres_sem_acento)))
    .withColumn("nome_argentina", upper(translate(col("nome_argentina"), caracteres_com_acento, caracteres_sem_acento)))
    .withColumn("nome_mexico", upper(translate(col("nome_mexico"), caracteres_com_acento, caracteres_sem_acento)))
    .select(
        "produto_id", "nome_interno", "nome_brasil", "nome_argentina",
        "nome_mexico", "nome_ingles", "tamanho", "sku",
        "preco_brasil_brl", "preco_argentina_ars", "preco_mexico_mxn"
    )
)

df_produtos_casted.display()


In [0]:
# SCD2 real via MERGE

from pyspark.sql.functions import current_date, lit
from delta.tables import DeltaTable

tabela_existe = spark.catalog.tableExists("poc_latam_food.silver.produtos")

if not tabela_existe:
    # Primeira execução: carga inicial, sem comparação
    df_primeira_carga = (
        df_produtos_casted
        .withColumn("data_inicio", current_date())
        .withColumn("data_fim", lit(None).cast("date"))
        .withColumn("flag_ativo", lit(True))
    )
    df_primeira_carga.write.format("delta").saveAsTable("poc_latam_food.silver.produtos")
    print("Carga inicial da Silver de Produtos concluída.")

else:
    tabela_silver = DeltaTable.forName(spark, "poc_latam_food.silver.produtos")

    colunas_comparadas = [
        "nome_brasil", "nome_argentina", "nome_mexico", "nome_ingles",
        "preco_brasil_brl", "preco_argentina_ars", "preco_mexico_mxn"
    ]
    condicao_mudanca = " OR ".join([
        f"target.{c} <> source.{c}" for c in colunas_comparadas
    ])

    # Passo 1: fechar versões desatualizadas
    (
        tabela_silver.alias("target")
        .merge(
            df_produtos_casted.alias("source"),
            "target.produto_id = source.produto_id AND target.flag_ativo = true"
        )
        .whenMatchedUpdate(
            condition=condicao_mudanca,
            set={"flag_ativo": lit(False), "data_fim": current_date()}
        )
        .execute()
    )

    # Passo 2: inserir novas versões (produtos novos ou alterados)
    df_vigentes = spark.table("poc_latam_food.silver.produtos").filter("flag_ativo = true")

    df_para_inserir = (
        df_produtos_casted.alias("source")
        .join(df_vigentes.alias("target"), on="produto_id", how="left_anti")
        .withColumn("data_inicio", current_date())
        .withColumn("data_fim", lit(None).cast("date"))
        .withColumn("flag_ativo", lit(True))
    )

    df_para_inserir.write.format("delta").mode("append").saveAsTable("poc_latam_food.silver.produtos")

    print(f"MERGE concluído: {df_para_inserir.count()} nova(s) versão(ões) inserida(s).")

In [0]:
# Validação visual - silver.produtos

spark.table("poc_latam_food.silver.produtos").display()

print(f"Total de produtos: {spark.table('poc_latam_food.silver.produtos').count()}")